# Round 5: representation + capacity + analysis-driven features - S6E10

Round 4 gave no signal (RF leaf=10 and depth=25 working-signal worse, LGBM lr=0.03 a tie). The
incumbent stays Exp 2 raw LightGBM at 0.95847 mean OOF AUC; the local target is 0.96. This round
tests three DIFFERENT kinds of levers, still on the existing data only:

- **Exp 15: random forest with ordinal-encoded categoricals** (Andrea's preferred family; the
  representation itself is the untested lever - one-hot spreads 4 low-cardinality cats into sparse
  binaries, ordinal keeps them dense and allows ordered split sequences).
- **Exp 16: LightGBM num_leaves=127** - capacity step directly on the incumbent (not gated on any
  prior winner; one knob from Exp 8).
- **Exp 17: LightGBM with one-hot categoricals** - native 'category' handling vs one-hot, same
  question on the GBM side.
- **Exp 18: LightGBM + interaction terms motivated by error analysis** - a NEW feature hypothesis,
  not the dropped round-2/3 set: first the incumbent's OOF predictions are recomputed and segmented
  to find where ranking is weakest; the interactions target the dominant-feature pairs.

Protocol identical to rounds 3-4 (same 5 folds, seed 42), and the incumbent is RECOMPUTED in-run so
paired deltas use same-run numbers (with a determinism check against the round-3 constants). This
also closes round 3's pooled-AUC gap: every config gets mean-of-folds AND pooled. All OOF
predictions are cached to the gitignored oof_cache/ directory (executor pattern from round 4).


In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']

# Round-3 incumbent (Exp 8) per-fold AUCs - determinism reference for the in-run recompute.
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]


In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])

def to_lgbm(df):
    # LightGBM-native copy: categoricals as 'category', NaN left alone. sklearn frames keep object dtype.
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def to_onehot(df):
    out = pd.get_dummies(df, columns=CATEGORICAL)
    bool_cols = [c for c in out.columns if out[c].dtype == bool]
    if bool_cols:
        out[bool_cols] = out[bool_cols].astype(int)
    return out

def rf_pipeline(train_frame, min_samples_leaf=5, n_estimators=400, representation='onehot'):
    numeric_columns = train_frame.columns.difference(CATEGORICAL).tolist()
    if representation == 'onehot':
        cat_steps = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                              ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    else:
        cat_steps = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                              ('ordinal', OrdinalEncoder())])
    preprocessor = ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), numeric_columns),
        ('categorical', cat_steps, CATEGORICAL),
    ])
    return Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=n_estimators, min_samples_leaf=min_samples_leaf,
            n_jobs=-1, random_state=RANDOM_STATE)),
    ])

def oof_lgbm(feat_frame, label, name, learning_rate=0.05, num_leaves=63, n_estimators=1000,
             native_cats=True, return_model=False):
    frame = to_lgbm(feat_frame) if native_cats else to_onehot(feat_frame)
    oof = np.zeros(len(frame))
    aucs, iters = [], []
    last_model = None
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(frame, label), 1):
        X_tr, y_tr = frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = frame.iloc[va_idx], label.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        model = lgb.LGBMClassifier(learning_rate=learning_rate, num_leaves=num_leaves,
                                   n_estimators=n_estimators, n_jobs=-1,
                                   random_state=RANDOM_STATE, verbose=-1)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        p = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = p
        a = roc_auc_score(y_va, p)
        aucs.append(a); iters.append(model.best_iteration_)
        last_model = model
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    if return_model:
        return oof, aucs, iters, last_model
    return oof, aucs, iters

def oof_rf(feat_frame, label, name, **rf_kwargs):
    oof = np.zeros(len(feat_frame))
    aucs = []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        rf = rf_pipeline(X_tr, **rf_kwargs)
        rf.fit(X_tr, y_tr)
        p = rf.predict_proba(X_va)[:, 1]
        oof[va_idx] = p
        a = roc_auc_score(y_va, p)
        aucs.append(a)
        print(f'  {name} fold {fold}: AUC {a:.5f}')
    return oof, aucs

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features')


Full train: 699,635 rows, 21 raw features


## Reference: recompute the incumbent in-run + error analysis

Same-run incumbent OOF (Exp 8 config) with a determinism check against the round-3 constants, its
pooled AUC (closes the round-3 gap), then the analysis that motivates Exp 18: where is the
incumbent's ranking weakest (by Class and Type of Travel), and which features dominate its splits.


In [3]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc, inc_model = oof_lgbm(features, target, 'INC', return_model=True)
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
print(f'  in-run folds: ' + ' '.join(f'{a:.5f}' for a in aucs_inc))
det_ok = max(diffs) < 1e-4
det_note = 'PASS - identical folds' if det_ok else 'WARN - treat paired deltas vs constants as approximate; using in-run folds'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc
INC_MEAN = float(np.mean(aucs_inc))
print(f'  incumbent mean OOF AUC: {INC_MEAN:.5f} +/- {np.std(aucs_inc):.5f} | pooled: {roc_auc_score(target, oof_inc):.6f}')

print()
print('Segment analysis (where is ranking weakest):')
analysis = pd.DataFrame({'y': target.to_numpy(), 'p': oof_inc}, index=features.index)
for col in ['Class', 'Type of Travel']:
    for level, idx in features.groupby(col).groups.items():
        sub = analysis.loc[idx]
        print(f'  {col} = {level}: n={len(sub):,}, base_rate={sub.y.mean():.3f}, AUC={roc_auc_score(sub.y, sub.p):.5f}')

print()
imp = pd.Series(inc_model.booster_.feature_importance(importance_type='gain'),
                index=inc_model.booster_.feature_name()).sort_values(ascending=False)
print('Top-8 gain features (last fold model):')
print(imp.head(8).to_string())


Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 1: AUC 0.95851 (best_iteration=296)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 2: AUC 0.95751 (best_iteration=265)


...

/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 4: AUC 0.95841 (best_iteration=183)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 5: AUC 0.95885 (best_iteration=555)
  in-run folds: 0.95851 0.95751 0.95907 0.95841 0.95885
  determinism vs round-3 constants: max abs diff 0.00000 (PASS - identical folds)
  incumbent mean OOF AUC: 0.95847 +/- 0.00053 | pooled: 0.958455

Segment analysis (where is ranking weakest):
  Class = Business: n=342,212, base_rate=0.725, AUC=0.93385
  Class = Eco: n=327,404, base_rate=0.168, AUC=0.90175
  Class = Eco Plus: n=30,019, base_rate=0.242, AUC=0.92773


  Type of Travel = Business travel: n=497,441, base_rate=0.584, AUC=0.95407
  Type of Travel = Personal Travel: n=202,194, base_rate=0.097, AUC=0.82637

Top-8 gain features (last fold model):
Online_boarding           2.512467e+06
Inflight_wifi_service     6.391398e+05
Type_of_Travel            5.922798e+05
Customer_Type             1.825365e+05
Inflight_entertainment    1.780257e+05
Class                     1.668600e+05
Checkin_service           1.090808e+05
Baggage_handling          7.857271e+04


## Exp 15: random forest, ordinal-encoded categoricals

One change from the Exp 10 RF config (400 trees, leaf=5): ordinal encoding instead of one-hot.
Hypothesis: with only 4 low-cardinality categoricals, one-hot creates sparse binary splits; ordinal
keeps each categorical as one dense feature, letting the tree make repeated ordered splits on it.
Representation is an untested RF lever - this is the round's RF-priority move.


In [4]:
print('Exp 15: RF 400 trees, leaf=5, ORDINAL categoricals')
oof15, aucs15 = oof_rf(features, target, 'E15', representation='ordinal')
print(f'Exp 15 mean OOF AUC: {np.mean(aucs15):.5f} +/- {np.std(aucs15):.5f} | pooled: {roc_auc_score(target, oof15):.6f}')


Exp 15: RF 400 trees, leaf=5, ORDINAL categoricals


  E15 fold 1: AUC 0.95669


  E15 fold 2: AUC 0.95572


  E15 fold 3: AUC 0.95700


  E15 fold 4: AUC 0.95685


  E15 fold 5: AUC 0.95652
Exp 15 mean OOF AUC: 0.95656 +/- 0.00045 | pooled: 0.956556


## Exp 16: LightGBM num_leaves=127

One knob from the incumbent: 63 -> 127 leaves, lr=0.05, early stop 50. Hypothesis: the gain ranking
is dominated by a few features whose interactions may need more splits; with 700k rows there is
plenty of data per leaf. Direct test on the incumbent - not gated on any prior winner.


In [5]:
print('Exp 16: LightGBM num_leaves=127, lr=0.05')
oof16, aucs16, iters16 = oof_lgbm(features, target, 'E16', num_leaves=127)
print(f'Exp 16 mean OOF AUC: {np.mean(aucs16):.5f} +/- {np.std(aucs16):.5f} | pooled: {roc_auc_score(target, oof16):.6f}')


Exp 16: LightGBM num_leaves=127, lr=0.05


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E16 fold 1: AUC 0.95851 (best_iteration=150)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E16 fold 2: AUC 0.95770 (best_iteration=196)


...

  E16 fold 3: AUC 0.95930 (best_iteration=210)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E16 fold 4: AUC 0.95835 (best_iteration=195)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E16 fold 5: AUC 0.95914 (best_iteration=178)
Exp 16 mean OOF AUC: 0.95860 +/- 0.00058 | pooled: 0.958586


## Exp 17: LightGBM with one-hot categoricals

One change from the incumbent: one-hot encoding instead of native 'category' handling. Hypothesis:
with only 4 low-cardinality cats, one-hot lets each level split independently rather than under
LightGBM's categorical partitioning; representation is as untested here as it was for the RF.


In [6]:
print('Exp 17: LightGBM one-hot cats, lr=0.05, num_leaves=63')
oof17, aucs17, iters17 = oof_lgbm(features, target, 'E17', native_cats=False)
print(f'Exp 17 mean OOF AUC: {np.mean(aucs17):.5f} +/- {np.std(aucs17):.5f} | pooled: {roc_auc_score(target, oof17):.6f}')


Exp 17: LightGBM one-hot cats, lr=0.05, num_leaves=63


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E17 fold 1: AUC 0.95851 (best_iteration=296)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E17 fold 2: AUC 0.95751 (best_iteration=265)


...

  E17 fold 3: AUC 0.95907 (best_iteration=212)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E17 fold 4: AUC 0.95841 (best_iteration=183)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E17 fold 5: AUC 0.95884 (best_iteration=575)
Exp 17 mean OOF AUC: 0.95847 +/- 0.00053 | pooled: 0.958453


## Exp 18: LightGBM + interaction terms from the error analysis

A NEW feature hypothesis (not the dropped round-2/3 aggregate set): three pairwise interactions
among the dominant features - `Online boarding x Inflight wifi service`, and each of those two
crossed with the travel-type/class codes. If the segment analysis above shows weak ranking inside
specific Class/Travel groups, interactions with those group codes are the direct mechanical response.


In [7]:
def add_interactions(df):
    out = df.copy()
    travel = out['Type of Travel'].astype('category').cat.codes
    cls = out['Class'].astype('category').cat.codes
    out['boarding_x_wifi'] = out['Online boarding'] * out['Inflight wifi service']
    out['boarding_x_travel'] = out['Online boarding'] * travel
    out['wifi_x_class'] = out['Inflight wifi service'] * cls
    return out

features_x = add_interactions(features)
print('Exp 18: LightGBM + 3 interaction terms, lr=0.05, num_leaves=63')
oof18, aucs18, iters18 = oof_lgbm(features_x, target, 'E18')
print(f'Exp 18 mean OOF AUC: {np.mean(aucs18):.5f} +/- {np.std(aucs18):.5f} | pooled: {roc_auc_score(target, oof18):.6f}')


Exp 18: LightGBM + 3 interaction terms, lr=0.05, num_leaves=63


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E18 fold 1: AUC 0.95831 (best_iteration=269)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E18 fold 2: AUC 0.95746 (best_iteration=293)


...

  E18 fold 3: AUC 0.95933 (best_iteration=385)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E18 fold 4: AUC 0.95848 (best_iteration=319)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E18 fold 5: AUC 0.95898 (best_iteration=430)
Exp 18 mean OOF AUC: 0.95851 +/- 0.00064 | pooled: 0.958507


## Results, verdicts, and refit guard

Paired per-fold deltas against the IN-RUN incumbent folds. Labels: WORKING SIGNAL when the mean
paired delta clears 2x the spread of the per-fold deltas (descriptive, not proof), NO SIGNAL
otherwise. A candidate earns a full-data refit + submission file only with a working signal over the
incumbent - and submitting stays Andrea's call.


In [8]:
def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return deltas, mean_d, spread, label

results = {}
for exp_id, name, aucs, oof in [
    ('15', 'RF ordinal cats', aucs15, oof15),
    ('16', 'LGBM num_leaves=127', aucs16, oof16),
    ('17', 'LGBM one-hot cats', aucs17, oof17),
    ('18', 'LGBM + interactions', aucs18, oof18),
]:
    deltas, mean_d, spread, label = paired_verdict(aucs, INC_FOLDS)
    results[exp_id] = dict(name=name, aucs=aucs, oof=oof, mean_d=mean_d, label=label)
    print(f'Exp {exp_id} {name}: mean {np.mean(aucs):.5f} +/- {np.std(aucs):.5f} | pooled {roc_auc_score(target, oof):.6f}')
    print(f'  paired deltas vs in-run incumbent: ' + ' '.join(f'{d:+.5f}' for d in deltas) + f' (mean {mean_d:+.5f})')
    print(f'  {label}')
    print()

best_oof = max([INC_MEAN] + [float(np.mean(r['aucs'])) for r in results.values()])
print(f'Best OOF mean incl. incumbent: {best_oof:.5f} - >=0.96: {best_oof >= 0.96} (descriptive; target claims stay cautious - repeated selection on this data)')

winners = [k for k, r in results.items()
           if r['mean_d'] > 0 and 'WORKING SIGNAL: better' in r['label']]
print()
if not winners:
    print('No candidate earned a refit: incumbent Exp 2 raw LightGBM stands, submission_lgbm.csv remains the best submission candidate.')
else:
    win = max(winners, key=lambda k: np.mean(results[k]['aucs']))
    print(f'Winner: Exp {win} ({results[win]["name"]}) - refits on all rows below; logged as a working signal, exploratory.')


Exp 15 RF ordinal cats: mean 0.95656 +/- 0.00045 | pooled 0.956556
  paired deltas vs in-run incumbent: -0.00182 -0.00179 -0.00207 -0.00156 -0.00233 (mean -0.00191)
  WORKING SIGNAL: worse (descriptive, not proof)



Exp 16 LGBM num_leaves=127: mean 0.95860 +/- 0.00058 | pooled 0.958586
  paired deltas vs in-run incumbent: +0.00000 +0.00018 +0.00022 -0.00006 +0.00029 (mean +0.00013)
  NO SIGNAL (within 2x per-fold delta spread - no action)

Exp 17 LGBM one-hot cats: mean 0.95847 +/- 0.00053 | pooled 0.958453
  paired deltas vs in-run incumbent: +0.00000 +0.00000 +0.00000 +0.00000 -0.00001 (mean -0.00000)
  NO SIGNAL (within 2x per-fold delta spread - no action)



Exp 18 LGBM + interactions: mean 0.95851 +/- 0.00064 | pooled 0.958507
  paired deltas vs in-run incumbent: -0.00020 -0.00006 +0.00026 +0.00008 +0.00013 (mean +0.00004)
  NO SIGNAL (within 2x per-fold delta spread - no action)

Best OOF mean incl. incumbent: 0.95860 - >=0.96: False (descriptive; target claims stay cautious - repeated selection on this data)

No candidate earned a refit: incumbent Exp 2 raw LightGBM stands, submission_lgbm.csv remains the best submission candidate.


In [9]:
# Refit ONLY fires on a working-signal winner over the standing incumbent.
if not winners:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    if win == '15':
        final = rf_pipeline(features, representation='ordinal')
        final.fit(features, target)
        test_probabilities = final.predict_proba(test_features)[:, 1]
    else:
        prep = {'16': (features, dict(num_leaves=127), True),
                '17': (features, dict(), False),
                '18': (features_x, dict(), True)}[win]
        frame_full, overrides, native = prep
        mean_iters = {'16': iters16, '17': iters17, '18': iters18}[win]
        final = lgb.LGBMClassifier(learning_rate=0.05, n_estimators=int(np.mean(mean_iters)),
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=-1, **overrides)
        frame = to_lgbm(frame_full) if native else to_onehot(frame_full)
        final.fit(frame, target)
        test_frame = add_interactions(test_features) if win == '18' else test_features
        test_frame = to_lgbm(test_frame) if native else to_onehot(test_frame)
        test_probabilities = final.predict_proba(test_frame)[:, 1]
    submission = pd.read_csv(data_dir / 'sample_submission.csv')
    submission['satisfaction'] = test_probabilities
    assert len(submission) == len(test)
    assert submission['id'].equals(test['id'])
    out_path = competition_dir / 'submission_round5.csv'
    submission.to_csv(out_path, index=False)
    print(f"Saved {len(submission):,} predictions to {out_path} - submitting stays Andrea's call.")


No submission file written this round.


In [10]:
# OOF cache (executor pattern from round 4): persist all predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round5_oof.npz',
         oof_inc=np.asarray(oof_inc), aucs_inc=np.asarray(aucs_inc), iters_inc=np.asarray(iters_inc),
         oof15=np.asarray(oof15), aucs15=np.asarray(aucs15),
         oof16=np.asarray(oof16), aucs16=np.asarray(aucs16), iters16=np.asarray(iters16),
         oof17=np.asarray(oof17), aucs17=np.asarray(aucs17), iters17=np.asarray(iters17),
         oof18=np.asarray(oof18), aucs18=np.asarray(aucs18), iters18=np.asarray(iters18),
         y=np.asarray(target).astype(int))
meta = {
    'round': 5,
    'protocol': '5-fold stratified OOF, seed 42; LightGBM inner-10% early stop per fold',
    'dataset': {'file': 'data/train.csv', 'rows': int(len(target)), 'sha256': h.hexdigest()},
    'configs': {
        'incumbent': 'LightGBM lr=0.05, num_leaves=63, n_estimators=1000, early stop 50 (Exp 8 config, recomputed in-run)',
        'exp15': 'RandomForest 400 trees, leaf=5, ordinal categoricals',
        'exp16': 'LightGBM num_leaves=127, lr=0.05',
        'exp17': 'LightGBM one-hot categoricals, lr=0.05, num_leaves=63',
        'exp18': 'LightGBM + boarding_x_wifi, boarding_x_travel, wifi_x_class; lr=0.05, num_leaves=63',
    },
    'saved_at': time.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
(cache_dir / 'round5_oof.json').write_text(_json.dumps(meta, indent=2))
print('OOF cache written to', cache_dir)


OOF cache written to oof_cache


## What happens next

- If a representation or capacity move shows a working signal: adopt it as reference and combine
  carefully (one knob at a time still) - e.g. num_leaves on top of the winning representation.
- If nothing shows a signal: remaining on-data levers are min_data_in_leaf / feature_fraction +
  bagging on the incumbent, the missing-value handling experiment, and heavier ensembling
  (multi-seed averaging) - each with diminishing expected returns and a real multiple-comparisons
  cost. The external-data question (original passenger-survey dataset) belongs to Andrea; its
  public provenance/terms can be inspected read-only first, but no use before his explicit OK.
- The honest headroom note: rounds 3-5 are measuring a plateau region around 0.9585 for single
  LightGBM configs on this data; expect small deltas from here, and weigh each new round against
  that evidence.
